In [1]:
import numpy as np 

In [2]:
def l2_np(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.sum((a - b) ** 2))

def l1_np(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.sum(np.abs(a - b)))

def cosine_distance_np(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    a_mag = np.linalg.norm(a)
    b_mag = np.linalg.norm(b)
    if a_mag == 0 or b_mag == 0:
        return 1.0
    return float(1 - np.dot(a, b) / (a_mag * b_mag))

def minkowski_distance_np(a, b, p=2):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    diff = np.abs(a - b)
    if p == float('inf'):
        return float(np.max(diff))
    return float(np.sum(diff ** p) ** (1 / p))

In [5]:
import numpy as np


class KNN:
    def __init__(self, k=5, distance_fn=None, weighted=False,
                 task="classification", p=2):
        self.k = k
        self.distance_fn = distance_fn
        self.weighted = weighted
        self.task = task
        self.p = p
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        dist = self._pairwise_distances(X, self.X_train)

        k = min(self.k, self.X_train.shape[0])
        nn_idx = np.argsort(dist, axis=1)[:, :k]
        nn_dist = np.take_along_axis(dist, nn_idx, axis=1)
        nn_lab = self.y_train[nn_idx]

        if self.task == "classification":
            return self._classify_batch(nn_dist, nn_lab)
        return self._regress_batch(nn_dist, nn_lab)

    def _pairwise_distances(self, A, B):
        if self.distance_fn is not None:
            return self.distance_fn(A, B)

        if self.p == 2:
            a2 = (A ** 2).sum(axis=1)[:, None]
            b2 = (B ** 2).sum(axis=1)[None, :]
            d2 = a2 + b2 - 2 * A @ B.T
            np.maximum(d2, 0, out=d2)
            return np.sqrt(d2)

        diff = np.abs(A[:, None, :] - B[None, :, :])
        if self.p == np.inf:
            return diff.max(axis=-1)
        return (diff ** self.p).sum(axis=-1) ** (1.0 / self.p)

    def _classify_batch(self, nn_dist, nn_lab):
        n_test, k = nn_lab.shape
        preds = np.empty(n_test, dtype=self.y_train.dtype)

        for i in range(n_test):
            labels, inv = np.unique(nn_lab[i], return_inverse=True)
            if self.weighted:
                w = 1.0 / (nn_dist[i] + 1e-10)
            else:
                w = np.ones(k)

            votes = np.zeros(len(labels))
            np.add.at(votes, inv, w)
            preds[i] = labels[votes.argmax()]
        return preds

    def _regress_batch(self, nn_dist, nn_lab):
        y = nn_lab.astype(float)
        if self.weighted:
            w = 1.0 / (nn_dist + 1e-10)
            w_sum = w.sum(axis=1)
            out = (w * y).sum(axis=1) / np.where(w_sum > 0, w_sum, 1.0)
            return np.where(w_sum > 0, out, 0.0)
        return y.mean(axis=1)

    def _predict_one(self, x):
        x = np.asarray(x, dtype=float).reshape(1, -1)
        return self.predict(x)[0]

    def predict_with_neighbors(self, x):
        x = np.asarray(x, dtype=float).reshape(1, -1)
        dist = self._pairwise_distances(x, self.X_train)[0]
        k = min(self.k, self.X_train.shape[0])
        nn_idx = np.argsort(dist)[:k]
        neighbors = [(float(dist[i]), int(i), self.y_train[i]) for i in nn_idx]
        prediction = self.predict(x)[0]
        return prediction, neighbors

In [6]:
def test_knn():
    # ---------- classification ----------
    X_tr = [[1], [2], [3], [4], [6], [7], [8], [9]]
    y_tr = [0, 0, 0, 0, 1, 1, 1, 1]

    m = KNN(k=3, task="classification")
    m.fit(X_tr, y_tr)
    print(m.predict([[0.5], [4.6], [5.4], [9.9]]))   # [0 0 1 1]
    print(m._predict_one([4.6]))                     # 0

    mw = KNN(k=3, weighted=True, task="classification")
    mw.fit(X_tr, y_tr)
    print(mw.predict([[4.6], [5.4]]))                # [0 1]

    # ---------- regression ----------
    X_r = [[1], [2], [3], [4], [5], [6], [7], [8]]
    y_r = [2, 4, 6, 8, 10, 12, 14, 16]

    r = KNN(k=3, task="regression")
    r.fit(X_r, y_r)
    print(r.predict([[4.5], [6.5]]))                 # [10. 14.]

    rw = KNN(k=3, weighted=True, task="regression")
    rw.fit(X_r, y_r)
    print(rw.predict([[4.5], [4.0]]))

    # ---------- multi-dim ----------
    X2 = [[0,0],[0,1],[1,0],[1,1],[10,10],[10,11],[11,10],[11,11]]
    y2 = ["A","A","A","A","B","B","B","B"]
    m2 = KNN(k=3, task="classification")
    m2.fit(X2, y2)
    print(m2.predict([[0.2,0.2],[10.5,10.5]]))       # ['A' 'B']

    # ---------- predict_with_neighbors ----------
    pred, nb = m.predict_with_neighbors([4.6])
    print(pred, nb)

    # ---------- batch == single ----------
    for x in [0.5, 4.6, 5.4, 9.9]:
        assert m._predict_one([x]) == m.predict([[x]])[0]

    print("all good")


if __name__ == "__main__":
    test_knn()

[0 0 1 1]
0
[0 1]
[10. 14.]
[9.42857143 8.        ]
['A' 'B']
0 [(0.5999999999999995, 3, np.int64(0)), (1.4000000000000004, 4, np.int64(1)), (1.5999999999999996, 2, np.int64(0))]
all good


In [3]:

class Simple_KNN:
    def __init__(self, k=3, distance_fn=None, task="classification"):
        self.k = k
        self.distance_fn = distance_fn or euclidean
        self.task = task

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)

    def predict(self, X):
        X = np.asarray(X, dtype=float)

        predictions = []

        for x in X:
            # Step 1: Calculate distance from x to every training sample
            distances = []

            for x_train in self.X_train:
                d = self.distance_fn(x, x_train)
                distances.append(d)

            # Step 2: Find indices of k nearest neighbors
            nearest = np.argsort(distances)[:self.k]

            # Step 3: Get their labels
            labels = self.y_train[nearest]

            # Step 4: Make a prediction
            if self.task == "classification":
                values, counts = np.unique(
                    labels, return_counts=True
                )
                prediction = values[np.argmax(counts)]

            else:
                prediction = np.mean(labels.astype(float))

            predictions.append(prediction)

        return np.array(predictions)
